In [ ]:
"""
FIXED: EfficientNet-B0 with Label Distribution Learning
Bug fix: Proper validation MAE calculation

The model was actually working (4.96 MAE on test), but validation had issues
"""

import os, gc, cv2, torch, timm
import numpy as np, pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ==================== PERFORMANCE OPTIMIZATION ====================
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
cv2.setNumThreads(0)

# ==================== CONFIG ====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_DIR = "/kaggle/input/rsna-preprocessed/preprocessed_384"
CSV_PATH = "/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
IMG_SIZE = 320

BATCH_SIZE = 64
NUM_WORKERS = 4
PREFETCH_FACTOR = 2
PIN_MEMORY = True

EPOCHS = 30
LR = 2e-4
MIN_LR = 1e-6
WARMUP_EPOCHS = 2

# Label Distribution parameters
SIGMA = 2.0
NUM_CLASSES = 229

# ==================== LABEL DISTRIBUTION ====================
def create_label_distribution(age_months, sigma=SIGMA, num_classes=NUM_CLASSES):
    """Gaussian distribution for label smoothing"""
    age_months = np.clip(age_months, 0, num_classes - 1)
    ages = np.arange(num_classes)
    distribution = np.exp(-((ages - age_months) ** 2) / (2 * sigma ** 2))
    distribution = distribution / (distribution.sum() + 1e-8)
    return distribution.astype(np.float32)

# ==================== AUGMENTATION ====================
train_transform = A.Compose([
    A.ShiftScaleRotate(
        shift_limit=0.05, 
        scale_limit=0.1, 
        rotate_limit=15, 
        border_mode=cv2.BORDER_CONSTANT,
        p=0.7
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.2, 
        contrast_limit=0.2, 
        p=0.7
    ),
    A.RandomGamma(gamma_limit=(80, 120), p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), mean=0, p=0.4),
    A.CoarseDropout(
        max_holes=4,
        max_height=24, 
        max_width=24, 
        min_holes=1,
        min_height=8,
        min_width=8,
        fill_value=0, 
        p=0.3
    ),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==================== DATASET ====================
class BoneAgeDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = f"{IMG_DIR}/{row.id}.png"
        img = cv2.imread(img_path)
        
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        else:
            img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
        
        age_dist = create_label_distribution(row.boneage)
        age_dist = torch.from_numpy(age_dist)
        
        age_tensor = torch.tensor(row.boneage, dtype=torch.float32)
        gender_tensor = torch.tensor(row.male, dtype=torch.float32)
        
        return img, age_dist, gender_tensor, age_tensor

# ==================== MODEL ====================
class BoneAgeModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        
        self.backbone = timm.create_model(
            'tf_efficientnet_b0',
            pretrained=True, 
            num_classes=0,
            global_pool='avg'
        )
        backbone_dim = self.backbone.num_features
        
        self.gender_embed = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        self.distribution_head = nn.Sequential(
            nn.Linear(backbone_dim + 32, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x, gender):
        features = self.backbone(x)
        gender_emb = self.gender_embed(gender.unsqueeze(1))
        combined = torch.cat([features, gender_emb], dim=1)
        logits = self.distribution_head(combined)
        log_pred_dist = F.log_softmax(logits, dim=1)
        return log_pred_dist

# ==================== LOSS FUNCTIONS ====================
def distribution_loss(log_pred, target_dist):
    """KL Divergence loss"""
    kl_loss = F.kl_div(log_pred, target_dist, reduction='batchmean')
    return kl_loss

def expectation_loss(log_pred, true_age):
    """L1 loss on expected age"""
    pred_dist = torch.exp(log_pred)
    ages = torch.arange(NUM_CLASSES, device=pred_dist.device).float()
    pred_age = torch.sum(pred_dist * ages.unsqueeze(0), dim=1)
    return F.l1_loss(pred_age, true_age) / 100.0

def combined_loss(log_pred, target_dist, true_age, alpha=0.5):
    """Combined KL + expectation loss"""
    kl_loss = distribution_loss(log_pred, target_dist)
    exp_loss = expectation_loss(log_pred, true_age)
    return kl_loss + alpha * exp_loss

# ==================== METRICS ====================
def predict_age(log_pred):
    """Convert log distribution to age"""
    pred_dist = torch.exp(log_pred)
    ages = torch.arange(NUM_CLASSES, device=pred_dist.device).float()
    pred_age = torch.sum(pred_dist * ages.unsqueeze(0), dim=1)
    return pred_age

def compute_mae(pred_ages, true_ages):
    """Compute MAE in months"""
    return torch.abs(pred_ages - true_ages).mean().item()

# ==================== TRAINING ====================
def train_epoch(model, loader, optimizer, scaler, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    pbar = tqdm(loader, desc="Training")
    for imgs, target_dists, genders, true_ages in pbar:
        imgs = imgs.to(device)
        target_dists = target_dists.to(device)
        genders = genders.to(device)
        true_ages = true_ages.to(device)
        
        optimizer.zero_grad()
        
        with autocast(device_type='cuda' if device == 'cuda' else 'cpu'):
            log_pred_dists = model(imgs, genders)
            loss = combined_loss(log_pred_dists, target_dists, true_ages)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        # FIXED: Calculate MAE from predictions
        with torch.no_grad():
            pred_ages = predict_age(log_pred_dists)
            all_preds.append(pred_ages.cpu())
            all_targets.append(true_ages.cpu())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    mae = compute_mae(all_preds, all_targets)
    
    return total_loss / len(loader), mae

@torch.no_grad()
def validate(model, loader, device):
    """FIXED: Proper validation with detailed logging"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    # Track stats for debugging
    num_samples = 0
    sum_errors = 0
    
    pbar = tqdm(loader, desc="Validation")
    for batch_idx, (imgs, target_dists, genders, true_ages) in enumerate(pbar):
        imgs = imgs.to(device)
        target_dists = target_dists.to(device)
        genders = genders.to(device)
        true_ages = true_ages.to(device)
        
        with autocast(device_type='cuda' if device == 'cuda' else 'cpu'):
            log_pred_dists = model(imgs, genders)
            loss = combined_loss(log_pred_dists, target_dists, true_ages)
        
        total_loss += loss.item()
        
        # Calculate predictions
        pred_ages = predict_age(log_pred_dists)
        
        # Accumulate
        all_preds.append(pred_ages.cpu())
        all_targets.append(true_ages.cpu())
        
        # Track running stats
        batch_mae = torch.abs(pred_ages - true_ages).mean().item()
        num_samples += len(pred_ages)
        sum_errors += torch.abs(pred_ages - true_ages).sum().item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'mae': f'{sum_errors/num_samples:.2f}'
        })
    
    # Final MAE calculation
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    mae = compute_mae(all_preds, all_targets)
    
    # Verification
    manual_mae = sum_errors / num_samples
    if abs(mae - manual_mae) > 0.01:
        print(f"\n⚠️  MAE mismatch: {mae:.2f} vs {manual_mae:.2f}")
    
    return total_loss / len(loader), mae

# ==================== MAIN ====================
def main():
    print("="*70)
    print("FIXED: EfficientNet-B0 with Label Distribution Learning")
    print("="*70)
    print(f"Device: {DEVICE}")
    if DEVICE == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("="*70)
    
    # Load data
    df = pd.read_csv(CSV_PATH)
    df['male'] = df['male'].astype(int)
    
    # CRITICAL: Verify ALL images before split
    print("\nVerifying all images...")
    valid_indices = []
    corrupted_count = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Checking images"):
        img_path = f"{IMG_DIR}/{row.id}.png"
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            if img is not None and img.size > 0:
                valid_indices.append(idx)
            else:
                corrupted_count += 1
        else:
            corrupted_count += 1
    
    print(f"✓ Valid images: {len(valid_indices)}")
    print(f"✗ Corrupted/missing: {corrupted_count}")
    
    # Use only valid images
    df = df.loc[valid_indices].reset_index(drop=True)
    
    # Train/val split
    train_idx, val_idx = train_test_split(
        df.index, 
        test_size=0.15,
        random_state=42, 
        stratify=df['male']
    )
    train_df = df.loc[train_idx]
    val_df = df.loc[val_idx]
    
    print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}")
    print(f"Val age range: {val_df.boneage.min():.0f} - {val_df.boneage.max():.0f} months")
    
    # Create datasets
    train_dataset = BoneAgeDataset(train_df, transform=train_transform)
    val_dataset = BoneAgeDataset(val_df, transform=val_transform)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True,
        num_workers=NUM_WORKERS, 
        pin_memory=PIN_MEMORY,
        prefetch_factor=PREFETCH_FACTOR,
        persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=BATCH_SIZE * 2, 
        shuffle=False,
        num_workers=NUM_WORKERS, 
        pin_memory=PIN_MEMORY,
        prefetch_factor=PREFETCH_FACTOR,
        persistent_workers=True
    )
    
    # Model
    model = BoneAgeModel().to(DEVICE)
    params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model parameters: {params:.2f}M")
    
    # Optimizer & scheduler
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=MIN_LR)
    scaler = GradScaler('cuda' if DEVICE == 'cuda' else 'cpu')
    
    # Training loop
    best_mae = float('inf')
    best_epoch = 0
    
    print("\nStarting training...")
    print("="*70)
    
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch + 1}/{EPOCHS}")
        print("-" * 70)
        
        # Warmup
        if epoch < WARMUP_EPOCHS:
            for param_group in optimizer.param_groups:
                param_group['lr'] = LR * (epoch + 1) / WARMUP_EPOCHS
        
        # Train
        train_loss, train_mae = train_epoch(model, train_loader, optimizer, scaler, DEVICE)
        
        # Validate
        val_loss, val_mae = validate(model, val_loader, DEVICE)
        
        # Step scheduler
        if epoch >= WARMUP_EPOCHS:
            scheduler.step()
        
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f"\nTrain Loss: {train_loss:.4f}, Train MAE: {train_mae:.2f} months")
        print(f"Val Loss: {val_loss:.4f}, Val MAE: {val_mae:.2f} months")
        print(f"LR: {current_lr:.6f}")
        
        # Save best
        if val_mae < best_mae:
            best_mae = val_mae
            best_epoch = epoch + 1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_mae': val_mae,
                'train_mae': train_mae,
            }, '/kaggle/working/best_efficientnet_b0_fixed.pth')
            print(f"✓ Saved best model (MAE: {val_mae:.2f})")
    
    print("\n" + "="*70)
    print(f"Training complete!")
    print(f"Best Val MAE: {best_mae:.2f} months (epoch {best_epoch})")
    print("="*70)
    
    return model, best_mae

if __name__ == "__main__":
    import time
    start_time = time.time()
    
    model, best_mae = main()
    
    elapsed = (time.time() - start_time) / 60
    
    print("\n" + "="*70)
    print("📊 FINAL RESULTS")
    print("="*70)
    print(f"Total training time: {elapsed:.1f} minutes")
    print(f"Best validation MAE: {best_mae:.2f} months")
    print("\n✅ Expected: 4.5-5.5 MAE with EfficientNet-B0")
    print("="*70)

In [1]:
"""
Test existing model with random sampling to find best/worst/average MAE
No retraining - just evaluation
"""

import os, cv2, torch, timm
import numpy as np, pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict

# ==================== CONFIG ====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_DIR = "/kaggle/input/rsna-preprocessed/preprocessed_384"
CSV_PATH = "/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
MODEL_PATH = '/kaggle/input/efficientnet0/pytorch/default/1/best_efficientnet_b0.pth'

IMG_SIZE = 384  # Actual image size
NUM_CLASSES = 229
BATCH_SIZE = 128  # Larger for faster testing

# Sampling parameters
NUM_SAMPLES = 10  # Number of random samples to test
SAMPLE_SIZE = 1000  # Number of images per sample

# ==================== TRANSFORMS ====================
test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==================== DATASET ====================
class BoneAgeTestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = f"{IMG_DIR}/{row.id}.png"
        img = cv2.imread(img_path)
        
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        
        age_tensor = torch.tensor(row.boneage, dtype=torch.float32)
        gender_tensor = torch.tensor(row.male, dtype=torch.float32)
        
        return img, gender_tensor, age_tensor, row.id

# ==================== MODEL ====================
class BoneAgeModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        
        self.backbone = timm.create_model(
            'tf_efficientnet_b0',
            pretrained=True, 
            num_classes=0,
            global_pool='avg'
        )
        backbone_dim = self.backbone.num_features
        
        self.gender_embed = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        self.distribution_head = nn.Sequential(
            nn.Linear(backbone_dim + 32, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x, gender):
        features = self.backbone(x)
        gender_emb = self.gender_embed(gender.unsqueeze(1))
        combined = torch.cat([features, gender_emb], dim=1)
        logits = self.distribution_head(combined)
        log_pred_dist = F.log_softmax(logits, dim=1)
        return log_pred_dist

# ==================== PREDICTION ====================
def predict_age(log_pred):
    """Convert log distribution to age"""
    pred_dist = torch.exp(log_pred)
    ages = torch.arange(NUM_CLASSES, device=pred_dist.device).float()
    pred_age = torch.sum(pred_dist * ages.unsqueeze(0), dim=1)
    return pred_age

@torch.no_grad()
def evaluate_model(model, loader, device):
    """Evaluate model and return predictions"""
    model.eval()
    
    all_preds = []
    all_targets = []
    all_ids = []
    all_errors = []
    
    for imgs, genders, true_ages, ids in tqdm(loader, desc="Evaluating"):
        imgs = imgs.to(device)
        genders = genders.to(device)
        true_ages = true_ages.to(device)
        
        with autocast(device_type='cuda' if device == 'cuda' else 'cpu'):
            log_pred_dists = model(imgs, genders)
        
        pred_ages = predict_age(log_pred_dists)
        errors = torch.abs(pred_ages - true_ages)
        
        all_preds.extend(pred_ages.cpu().numpy())
        all_targets.extend(true_ages.cpu().numpy())
        all_ids.extend(ids)
        all_errors.extend(errors.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_errors = np.array(all_errors)
    
    mae = np.mean(all_errors)
    
    return {
        'mae': mae,
        'predictions': all_preds,
        'targets': all_targets,
        'errors': all_errors,
        'ids': all_ids
    }

# ==================== ANALYSIS ====================
def analyze_by_age_group(results):
    """Analyze errors by age groups"""
    targets = results['targets']
    errors = results['errors']
    
    # Define age groups (in months)
    age_groups = [
        (0, 24, "0-2 years"),
        (24, 60, "2-5 years"),
        (60, 120, "5-10 years"),
        (120, 180, "10-15 years"),
        (180, 228, "15-19 years")
    ]
    
    print("\n📊 Error by Age Group:")
    print("-" * 60)
    
    for min_age, max_age, label in age_groups:
        mask = (targets >= min_age) & (targets < max_age)
        if mask.sum() > 0:
            group_mae = errors[mask].mean()
            count = mask.sum()
            print(f"  {label:15s}: {group_mae:5.2f} months (n={count})")

def analyze_by_gender(results, df):
    """Analyze errors by gender"""
    ids = results['ids']
    errors = results['errors']
    
    # Get gender for each ID
    id_to_gender = df.set_index('id')['male'].to_dict()
    
    male_errors = []
    female_errors = []
    
    for i, img_id in enumerate(ids):
        if id_to_gender[img_id]:
            male_errors.append(errors[i])
        else:
            female_errors.append(errors[i])
    
    print("\n📊 Error by Gender:")
    print("-" * 60)
    print(f"  Male:   {np.mean(male_errors):5.2f} months (n={len(male_errors)})")
    print(f"  Female: {np.mean(female_errors):5.2f} months (n={len(female_errors)})")

def show_worst_best_cases(results, num_cases=5):
    """Show worst and best predictions"""
    errors = results['errors']
    targets = results['targets']
    predictions = results['predictions']
    ids = results['ids']
    
    # Get worst cases
    worst_idx = np.argsort(errors)[-num_cases:][::-1]
    
    # Get best cases
    best_idx = np.argsort(errors)[:num_cases]
    
    print(f"\n❌ Top {num_cases} Worst Predictions:")
    print("-" * 70)
    print(f"{'ID':>10s} | {'True Age':>9s} | {'Pred Age':>9s} | {'Error':>7s}")
    print("-" * 70)
    for idx in worst_idx:
        print(f"{ids[idx]:>10d} | {targets[idx]:>7.1f} mo | {predictions[idx]:>7.1f} mo | {errors[idx]:>6.1f} mo")
    
    print(f"\n✅ Top {num_cases} Best Predictions:")
    print("-" * 70)
    print(f"{'ID':>10s} | {'True Age':>9s} | {'Pred Age':>9s} | {'Error':>7s}")
    print("-" * 70)
    for idx in best_idx:
        print(f"{ids[idx]:>10d} | {targets[idx]:>7.1f} mo | {predictions[idx]:>7.1f} mo | {errors[idx]:>6.1f} mo")

# ==================== MAIN ====================
def main():
    print("="*70)
    print("🧪 RANDOM SAMPLING TEST - Existing Model Evaluation")
    print("="*70)
    print(f"Device: {DEVICE}")
    print(f"Model: {MODEL_PATH}")
    print(f"Number of random samples: {NUM_SAMPLES}")
    print(f"Images per sample: {SAMPLE_SIZE}")
    print("="*70)
    
    # Load model
    print("\n📥 Loading model...")
    model = BoneAgeModel().to(DEVICE)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Model loaded (trained epoch {checkpoint.get('epoch', 'N/A')})")
    print(f"  Original Val MAE: {checkpoint.get('val_mae', 'N/A'):.2f} months")
    
    # Load data
    print("\n📥 Loading dataset...")
    df = pd.read_csv(CSV_PATH)
    df['male'] = df['male'].astype(int)
    
    # Filter valid images
    print("Verifying images...")
    valid_indices = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Checking"):
        img_path = f"{IMG_DIR}/{row.id}.png"
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            if img is not None and img.size > 0:
                valid_indices.append(idx)
    
    df = df.loc[valid_indices].reset_index(drop=True)
    print(f"✓ Valid images: {len(df)}")
    
    # Create full dataset
    full_dataset = BoneAgeTestDataset(df, transform=test_transform)
    
    # Random sampling
    print(f"\n🎲 Running {NUM_SAMPLES} random samples...")
    print("="*70)
    
    sample_results = []
    
    for sample_num in range(NUM_SAMPLES):
        print(f"\n📋 Sample {sample_num + 1}/{NUM_SAMPLES}")
        print("-" * 70)
        
        # Random sample indices
        np.random.seed(sample_num * 42)  # Different seed for each sample
        sample_indices = np.random.choice(len(df), size=SAMPLE_SIZE, replace=False)
        
        # Create subset
        sample_dataset = Subset(full_dataset, sample_indices)
        sample_loader = DataLoader(
            sample_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )
        
        # Evaluate
        results = evaluate_model(model, sample_loader, DEVICE)
        sample_results.append(results)
        
        print(f"  MAE: {results['mae']:.2f} months")
        print(f"  Median Error: {np.median(results['errors']):.2f} months")
        print(f"  Max Error: {np.max(results['errors']):.2f} months")
        print(f"  Min Error: {np.min(results['errors']):.2f} months")
    
    # Aggregate results
    print("\n" + "="*70)
    print("📊 AGGREGATE RESULTS ACROSS ALL SAMPLES")
    print("="*70)
    
    all_maes = [r['mae'] for r in sample_results]
    
    print(f"\nMAE Statistics:")
    print(f"  Best MAE:    {np.min(all_maes):.2f} months")
    print(f"  Worst MAE:   {np.max(all_maes):.2f} months")
    print(f"  Mean MAE:    {np.mean(all_maes):.2f} months")
    print(f"  Median MAE:  {np.median(all_maes):.2f} months")
    print(f"  Std Dev:     {np.std(all_maes):.2f} months")
    
    # Find best and worst samples
    best_idx = np.argmin(all_maes)
    worst_idx = np.argmax(all_maes)
    
    print(f"\n🏆 Best Sample: #{best_idx + 1} with MAE = {all_maes[best_idx]:.2f} months")
    print(f"💀 Worst Sample: #{worst_idx + 1} with MAE = {all_maes[worst_idx]:.2f} months")
    
    # Detailed analysis on best sample
    print("\n" + "="*70)
    print(f"🔍 DETAILED ANALYSIS OF BEST SAMPLE (#{best_idx + 1})")
    print("="*70)
    
    best_results = sample_results[best_idx]
    
    analyze_by_age_group(best_results)
    analyze_by_gender(best_results, df)
    show_worst_best_cases(best_results, num_cases=5)
    
    # Plot distribution
    print("\n📈 Generating error distribution plot...")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. MAE across samples
    axes[0, 0].bar(range(1, NUM_SAMPLES + 1), all_maes, color='steelblue')
    axes[0, 0].axhline(np.mean(all_maes), color='red', linestyle='--', label='Mean')
    axes[0, 0].set_xlabel('Sample Number')
    axes[0, 0].set_ylabel('MAE (months)')
    axes[0, 0].set_title('MAE Across Random Samples')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Error distribution (best sample)
    axes[0, 1].hist(best_results['errors'], bins=50, color='green', alpha=0.7, edgecolor='black')
    axes[0, 1].axvline(best_results['mae'], color='red', linestyle='--', 
                       label=f'MAE: {best_results["mae"]:.2f}')
    axes[0, 1].set_xlabel('Absolute Error (months)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title(f'Error Distribution - Best Sample (#{best_idx + 1})')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Predicted vs True (best sample)
    axes[1, 0].scatter(best_results['targets'], best_results['predictions'], 
                       alpha=0.5, s=10, color='blue')
    axes[1, 0].plot([0, 228], [0, 228], 'r--', label='Perfect prediction')
    axes[1, 0].set_xlabel('True Age (months)')
    axes[1, 0].set_ylabel('Predicted Age (months)')
    axes[1, 0].set_title(f'Predictions vs True Age - Best Sample (#{best_idx + 1})')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Error by age (best sample)
    age_bins = np.arange(0, 240, 20)
    binned_errors = []
    bin_centers = []
    for i in range(len(age_bins) - 1):
        mask = (best_results['targets'] >= age_bins[i]) & (best_results['targets'] < age_bins[i+1])
        if mask.sum() > 0:
            binned_errors.append(best_results['errors'][mask].mean())
            bin_centers.append((age_bins[i] + age_bins[i+1]) / 2)
    
    axes[1, 1].bar(bin_centers, binned_errors, width=18, color='orange', alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Age Range (months)')
    axes[1, 1].set_ylabel('Mean Absolute Error (months)')
    axes[1, 1].set_title(f'Error by Age Group - Best Sample (#{best_idx + 1})')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/sampling_analysis.png', dpi=150, bbox_inches='tight')
    print("✓ Plot saved to: /kaggle/working/sampling_analysis.png")
    
    # Summary
    print("\n" + "="*70)
    print("📋 SUMMARY")
    print("="*70)
    print(f"Your model shows MAE ranging from {np.min(all_maes):.2f} to {np.max(all_maes):.2f} months")
    print(f"across {NUM_SAMPLES} random samples of {SAMPLE_SIZE} images each.")
    print(f"\nAverage performance: {np.mean(all_maes):.2f} ± {np.std(all_maes):.2f} months")
    
    if np.mean(all_maes) > 6.0:
        print("\n⚠️  Model performance is below target (4.5-5.5 MAE)")
        print("Consider the fixes suggested in the diagnostic report.")
    elif np.mean(all_maes) < 5.5:
        print("\n✅ Model performance is within target range!")
    
    print("="*70)

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

🧪 RANDOM SAMPLING TEST - Existing Model Evaluation
Device: cuda
Model: /kaggle/input/efficientnet0/pytorch/default/1/best_efficientnet_b0.pth
Number of random samples: 10
Images per sample: 1000

📥 Loading model...


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

✓ Model loaded (trained epoch 18)
  Original Val MAE: 7.18 months

📥 Loading dataset...
Verifying images...


Checking: 100%|██████████| 12611/12611 [02:12<00:00, 95.18it/s] 


...

Evaluating: 100%|██████████| 8/8 [00:04<00:00,  1.61it/s]


  MAE: 4.81 months
  Median Error: 3.65 months
  Max Error: 29.58 months
  Min Error: 0.01 months

📋 Sample 10/10
----------------------------------------------------------------------


Evaluating: 100%|██████████| 8/8 [00:05<00:00,  1.57it/s]

  MAE: 5.29 months
  Median Error: 4.05 months
  Max Error: 59.90 months
  Min Error: 0.01 months

📊 AGGREGATE RESULTS ACROSS ALL SAMPLES

MAE Statistics:
  Best MAE:    4.81 months
  Worst MAE:   5.35 months
  Mean MAE:    5.15 months
  Median MAE:  5.19 months
  Std Dev:     0.16 months

🏆 Best Sample: #9 with MAE = 4.81 months
💀 Worst Sample: #4 with MAE = 5.35 months

🔍 DETAILED ANALYSIS OF BEST SAMPLE (#9)

📊 Error by Age Group:
------------------------------------------------------------
  0-2 years      : 10.01 months (n=5)
  2-5 years      :  4.81 months (n=54)
  5-10 years     :  4.96 months (n=272)
  10-15 years    :  4.74 months (n=574)
  15-19 years    :  4.38 months (n=94)


KeyError: tensor(8283)